# B03 V2 — Benchmark Teacher–Student dengan dua tahap training

**Versi eksperimen terpisah dari B03 V1. Jalankan setelah B02C berstatus READY.**

Pada V1, backbone langsung dilatih bersama classifier yang baru diinisialisasi. V2 menguji pendekatan dua tahap: pelajari classifier dahulu, kemudian fine-tune backbone dengan learning rate lebih rendah. Ini hipotesis perbaikan optimisasi; belum terbukti memperbaiki collapse pada data Anda.

| Tahap | Bagian yang dilatih | Batas epoch | Learning rate |
|---|---|---:|---:|
| Head | Classifier; backbone dibekukan | 5 tetap | 0.001 |
| Fine-tuning | Backbone dan classifier; BatchNorm tetap dibekukan | 25; patience 5 | 0.00001 |

Checkpoint head terbaik menjadi titik awal fine-tuning. Optimizer dibuat ulang untuk fine-tuning. Checkpoint akhir dipilih dari kedua tahap menggunakan **validation macro-F1**, lalu **recall kelas positif** sebagai pemecah seri. Jika keduanya sama, checkpoint head dipertahankan. Threshold tetap 0.5.

**Urutan pemakaian:** jalankan semua cell pada CBIS-DDSM untuk `mobilenet_v2` dan `vgg16` dahulu, lalu kirim ZIP V2 untuk ditinjau. Jika V2 dipakai sebagai protokol benchmark, jalankan seluruh delapan model menggunakan V2 pada setiap dataset. Hasil V1 tetap dilaporkan sebagai screening/protokol awal; jangan menyusun peringkat dengan mencampurkan model V1 dan V2.

| Kandidat Teacher | Kandidat Student |
|---|---|
| VGG16 | DenseNet121 |
| ResNet50 | MobileNetV2 |
| InceptionV3 | MobileNetV3-Large (`mobilenet_v3`) |
| RegNetY-8GF (`RegNetY080`) | NASNetMobile |

Ini benchmark supervised sebelum MoE/KD. Default satu seed 42 adalah screening, bukan bukti kestabilan multiseed. V2 mengubah warm-up dan learning rate sekaligus, sehingga peningkatan tidak dapat diatribusikan ke satu perubahan saja.

## 1. Runtime dan Google Drive
Gunakan GPU Colab. Tetap gunakan `tf_keras` secara konsisten, float32, dan BatchNorm non-fused. Cell memasangkan versi minor TF-Keras dengan TensorFlow bawaan tanpa mengganti TensorFlow/CUDA. Jika runtime sudah menjalankan konfigurasi yang berbeda, mulai sesi baru.

In [ ]:
import os,sys,subprocess,importlib.metadata as metadata
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'

import tensorflow as tf
tf_minor='.'.join(tf.__version__.split('.')[:2])

try: installed=metadata.version('tf-keras')
except metadata.PackageNotFoundError: installed=''
if '.'.join(installed.split('.')[:2])!=tf_minor:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={tf_minor}.0'])

import tf_keras as keras
print('TensorFlow:',tf.__version__,'TF-Keras:',keras.__version__)
gpu=tf.config.list_physical_devices('GPU')
if not gpu: raise RuntimeError('Select the GPU runtime for B03; B02 only requires the CPU.')
for device in gpu: tf.config.experimental.set_memory_growth(device,True)
keras.mixed_precision.set_global_policy('float32')
tf.config.experimental.enable_op_determinism()
print('GPU:',gpu)

from google.colab import drive
drive.mount('/content/drive')

## 2. Trainer V2
Sumber trainer disimpan sebagai teks lalu dijalankan. SHA-256 dihitung dari **teks yang benar-benar dijalankan** dan disimpan bersama konfigurasi; perubahan kode menghasilkan folder protokol baru. Jangan menimpa fungsi trainer di cell lain setelah cell ini dijalankan.

In [ ]:
TRAINER_SOURCE = r"""
import os, io, gc, json, time, hashlib, zipfile, shutil, platform

from pathlib import Path

import numpy as np

import pandas as pd

import tensorflow as tf

import tf_keras as keras

from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix, balanced_accuracy_score)

REGISTRY = {
    'vgg16': ('teacher','VGG16','vgg16'),
    'resnet50': ('teacher','ResNet50','resnet'),
    'inception_v3': ('teacher','InceptionV3','inception_v3'),
    'regnet_y_8gf': ('teacher','RegNetY080',None),
    'densenet121': ('student','DenseNet121','densenet'),
    'mobilenet_v2': ('student','MobileNetV2','mobilenet_v2'),
    'mobilenet_v3': ('student', 'MobileNetV3Large', None),
    'nasnet_mobile': ('student','NASNetMobile','nasnet'),
}

def json_safe(value):
    if isinstance(value, dict): return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value, (tuple,list)): return [json_safe(v) for v in value]
    if isinstance(value, np.generic): return json_safe(value.item())
    if isinstance(value, float) and not np.isfinite(value): return None
    return value

def save_json(path, obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(json_safe(obj),indent=2),encoding='utf-8'); os.replace(tmp,path)

def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

def metrics(y, p):
    y=np.asarray(y,dtype=int); p=np.asarray(p,dtype=float).reshape(-1)
    if len(y)!=len(p) or not np.isfinite(p).all() or ((p<0)|(p>1)).any():
        raise ValueError('Invalid predictions')
    pred=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(precision=precision_score(y,pred,zero_division=0),
        recall=recall_score(y,pred,zero_division=0), f1=f1_score(y,pred,zero_division=0),
        macro_f1=f1_score(y,pred,labels=[0,1],average='macro',zero_division=0),
        accuracy=accuracy_score(y,pred), balanced_accuracy=balanced_accuracy_score(y,pred),
        specificity=tn/(tn+fp) if tn+fp else np.nan,
        roc_auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
        average_precision=average_precision_score(y,p) if len(np.unique(y))==2 else np.nan,
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),threshold=.5,images=len(y))

def make_dataset(frame, image_dir, key, batch, seed, epoch=0, training=False, size=224):
    paths=[str(image_dir/p) for p in frame.cache_member]
    labels=frame.label.to_numpy(dtype=np.float32)
    d=tf.data.Dataset.from_tensor_slices((paths,labels))
    if training:
        d=d.shuffle(len(frame),seed=seed+epoch,reshuffle_each_iteration=False)
    preprocess_module=REGISTRY[key][2]
    def load(index, pair):
        path,label=pair
        x=tf.image.decode_png(tf.io.read_file(path),channels=3)
        x=tf.ensure_shape(tf.cast(x,tf.float32),[size,size,3])
        if training:
            # Stateless horizontal flip: image identity is not a laterality target.
            x=tf.image.stateless_random_flip_left_right(x,seed=tf.stack([
                tf.cast(seed+epoch*1009,tf.int32),tf.cast(index,tf.int32)]))
        if preprocess_module:
            x=getattr(keras.applications,preprocess_module).preprocess_input(x)
        # MobileNetV3 / RegNet contain their own rescaling and consume [0,255].
        return x,tf.reshape(label,[1])
    options=tf.data.Options(); options.experimental_deterministic=True
    options.threading.private_threadpool_size=2
    return d.enumerate().map(load,num_parallel_calls=2,deterministic=True).batch(batch).with_options(options).prefetch(1)

def load_ready_dataset(ready_path, local_root):
    ready_path=Path(ready_path); meta=json.loads(ready_path.read_text()); folder=ready_path.parent
    for filename,digest in meta['file_sha256'].items():
        if file_sha(folder/filename)!=digest: raise RuntimeError('B02 cache changed: '+filename)
    frames={s:pd.read_csv(folder/(s+'.csv'),dtype={'patient_id':str,'sample_id':str})
            for s in ['train','validation','test']}
    all_d=pd.concat(frames.values(),ignore_index=True)
    assert not all_d.sample_id.duplicated().any()
    assert all_d.groupby('patient_id').split.nunique().max()==1
    assert all(set(d.label)=={0,1} for d in frames.values())
    image_dir=Path(local_root)/meta['dataset']/meta['cohort_sha256'][:16]
    image_dir.mkdir(parents=True,exist_ok=True)
    marker=image_dir/'EXTRACTED.json'
    expected=meta['file_sha256']['images.zip']
    valid=marker.exists() and json.loads(marker.read_text()).get('sha256')==expected
    if not valid or not all((image_dir/f).is_file() for f in all_d.cache_member):
        local_zip=image_dir/'images.zip'
        shutil.copyfile(folder/'images.zip',local_zip)
        if file_sha(local_zip)!=expected: raise RuntimeError('Local cache copy corrupted')
        with zipfile.ZipFile(local_zip) as z:
            for member in all_d.cache_member:
                if Path(member).name!=member: raise ValueError('Unsafe cache member')
                (image_dir/member).write_bytes(z.read(member))
        save_json(marker,{'sha256':expected})
    # Audit the actual extracted model inputs, including reused local cache.
    from PIL import Image
    for row in all_d.itertuples():
        with Image.open(image_dir/row.cache_member) as im:
            a=np.asarray(im)
            if a.shape!=(meta['image_size'],meta['image_size']): raise ValueError('Unexpected image size')
            if hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:
                raise RuntimeError('Corrupted local image cache')
    return frames,image_dir,meta

def export_predictions(model, frame, ds, out, split, key, dataset, seed):
    probability=model.predict(ds,verbose=0).reshape(-1)
    result=metrics(frame.label.to_numpy(),probability)
    table=frame[['sample_id','patient_id','label']].copy()
    table['probability']=probability.astype(np.float64)
    table['prediction']=(probability>=.5).astype(int)
    table['model'],table['dataset'],table['training_seed']=key,dataset,seed
    table.to_csv(out/(split+'_predictions.csv'),index=False)
    result.update(model=key,dataset=dataset,training_seed=seed,split=split,role=REGISTRY[key][0])
    save_json(out/(split+'_metrics.json'),result)
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    fig,ax=plt.subplots(figsize=(5,4.5))
    names=['BI-RADS 1/2','BI-RADS 5/6'] if dataset=='inbreast' else ['Benign','Malignant']
    ConfusionMatrixDisplay(np.array([[result['tn'],result['fp']],[result['fn'],result['tp']]]),display_labels=names).plot(
        ax=ax,cmap='Blues',values_format='d',im_kw={'vmin':0,'vmax':int(frame.label.value_counts().max())})
    ax.set_title(f'{key} | {dataset}\n{split}; seed {seed}; threshold 0.5')
    fig.tight_layout(); fig.savefig(out/(split+'_confusion_matrix.png'),dpi=180);plt.close(fig)
    return result

def build_model(key, size=224, seed=42, weights='imagenet', learning_rate=1e-5, stage='finetune'):
    keras.utils.set_random_seed(seed)
    _, constructor, _ = REGISTRY[key]
    base = getattr(keras.applications, constructor)(include_top=False, weights=weights,
        input_shape=(size,size,3), pooling='avg')
    base.trainable = stage == 'finetune'
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
            layer.fused = False
    inp = keras.Input((size,size,3), name='preprocessed_image')
    x = base(inp, training=False)
    x = keras.layers.Dropout(.2, seed=seed, name='head_dropout')(x)
    output = keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='probability')(x)
    model = keras.Model(inp, output, name=key)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy())
    model.optimizer.build(model.trainable_variables)
    return model

def weight_snapshot(model):
    # Explicitly exclude optimizer: head and fine-tuning have different variable sets.
    base = next(layer for layer in model.layers if isinstance(layer, keras.Model))
    return tf.train.Checkpoint(backbone=base, classifier=model.get_layer('probability'))

def protocol_dir(run_root, dataset, protocol):
    signature = hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()[:12]
    return Path(run_root)/dataset/signature

def run_dir(run_root, dataset, key, protocol):
    return protocol_dir(run_root,dataset,protocol)/key/f"seed_{protocol['training_seed']}"

def experiment_config(key, dataset, protocol):
    return dict(protocol, model=key, dataset=dataset, role=REGISTRY[key][0],
        constructor=REGISTRY[key][1], preprocessing_module=REGISTRY[key][2])

def probability_diagnostics(y, p):
    p=np.asarray(p,dtype=np.float64).reshape(-1)
    q=np.clip(p,1e-7,1-1e-7); y=np.asarray(y,dtype=np.float64)
    return dict(probability_min=float(p.min()), probability_max=float(p.max()),
        probability_std=float(p.std()), positive_prediction_fraction=float((p>=.5).mean()),
        validation_bce_unweighted=float(-(y*np.log(q)+(1-y)*np.log(1-q)).mean()))

def preprocessing_audit(frames, image_dir, meta, key, protocol):
    from PIL import Image
    subset=frames['train'].iloc[:protocol['batch_size']]
    raw=np.stack([np.asarray(Image.open(image_dir/name).convert('RGB'),dtype=np.float32)
                  for name in subset.cache_member])
    ds=make_dataset(subset,image_dir,key,protocol['batch_size'],protocol['training_seed'],size=meta['image_size'])
    x,y=next(iter(ds)); actual=x.numpy()
    if key in ['vgg16','resnet50']:
        expected=raw[...,::-1]-np.array([103.939,116.779,123.68],dtype=np.float32)
    elif key=='densenet121':
        expected=(raw/255.-np.array([.485,.456,.406],dtype=np.float32))/np.array([.229,.224,.225],dtype=np.float32)
    elif key in ['inception_v3','mobilenet_v2','nasnet_mobile']:
        expected=raw/127.5-1.
    else: expected=raw  # RegNet and MobileNetV3 use internal preprocessing.
    np.testing.assert_allclose(actual,expected,rtol=1e-6,atol=2e-5)
    np.testing.assert_equal(y.numpy().reshape(-1),subset.label.to_numpy())
    return dict(model=key,constructor=REGISTRY[key][1],preprocessing=REGISTRY[key][2],
        checked_split='train',checked_images=len(subset),input_min=float(actual.min()),
        input_max=float(actual.max()),input_std=float(actual.std()),formula_check='passed')

def environment_record():
    return dict(python=platform.python_version(),tensorflow=tf.__version__,tf_keras=keras.__version__,
        gpu=[tf.config.experimental.get_device_details(g) for g in tf.config.list_physical_devices('GPU')])

def run_stage(key,frames,image_dir,meta,out,protocol,stage,initial_checkpoint=None):
    out.mkdir(parents=True,exist_ok=True)
    complete=out/'COMPLETE.json'
    if complete.exists():
        result=json.loads((out/'validation_metrics.json').read_text())
        state=json.loads(complete.read_text())
        return result,state
    keras.backend.clear_session();gc.collect()
    head=stage=='head'
    budget=protocol['head_epochs'] if head else protocol['finetune_epochs']
    learning_rate=protocol['head_learning_rate'] if head else protocol['finetune_learning_rate']
    latest=tf.train.latest_checkpoint(str(out/'last'))
    model=build_model(key,meta['image_size'],protocol['training_seed'],
        weights='imagenet' if head and not latest else None, learning_rate=learning_rate,stage=stage)
    selected_weights=weight_snapshot(model)
    if not head and not latest:
        if not initial_checkpoint: raise RuntimeError('Fine-tuning requires the selected head-stage weights')
        selected_weights.restore(initial_checkpoint).assert_existing_objects_matched()
    epoch_var=tf.Variable(0,dtype=tf.int64,trainable=False)
    best_f1=tf.Variable(-1.,dtype=tf.float64,trainable=False)
    best_recall=tf.Variable(-1.,dtype=tf.float64,trainable=False)
    wait=tf.Variable(0,dtype=tf.int64,trainable=False)
    best_epoch=tf.Variable(0,dtype=tf.int64,trainable=False)
    best_path=tf.Variable('',dtype=tf.string,trainable=False)
    history_var=tf.Variable('[]',dtype=tf.string,trainable=False)
    ckpt=tf.train.Checkpoint(model=model,optimizer=model.optimizer,epoch=epoch_var,
        best_f1=best_f1,best_recall=best_recall,wait=wait,best_epoch=best_epoch,
        best_path=best_path,history=history_var)
    manager=tf.train.CheckpointManager(ckpt,str(out/'last'),max_to_keep=2)
    best_manager=tf.train.CheckpointManager(selected_weights,str(out/'best'),max_to_keep=2)
    if latest:
        ckpt.restore(latest).assert_existing_objects_matched()
        print(key,stage,'resume after phase epoch',int(epoch_var.numpy()),flush=True)
    train=frames['train']; val=frames['validation']
    history=json.loads(history_var.numpy().decode())
    counts=train.label.value_counts().to_dict()
    weights={int(k):len(train)/(2*v) for k,v in counts.items()}
    val_ds=make_dataset(val,image_dir,key,protocol['batch_size'],protocol['training_seed'],size=meta['image_size'])
    if not latest:
        p0=model.predict(val_ds,verbose=0).reshape(-1)
        save_json(out/'initial_validation_diagnostics.json',dict(metrics(val.label,p0),
            **probability_diagnostics(val.label,p0),stage=stage,
            note='diagnostic only; not a candidate checkpoint'))
    env=environment_record()
    for epoch in range(int(epoch_var.numpy()),budget):
        if not head and int(wait.numpy())>=protocol['patience']: break
        global_epoch=epoch if head else protocol['head_epochs']+epoch
        train_ds=make_dataset(train,image_dir,key,protocol['batch_size'],protocol['training_seed'],
            epoch=global_epoch,training=True,size=meta['image_size'])
        start=time.perf_counter()
        fit=model.fit(train_ds,epochs=1,class_weight=weights,verbose=2,callbacks=[keras.callbacks.TerminateOnNaN()])
        train_seconds=time.perf_counter()-start
        loss=float(fit.history['loss'][0])
        if not np.isfinite(loss): raise RuntimeError('Nonfinite loss; no checkpoint committed')
        start=time.perf_counter();p=model.predict(val_ds,verbose=0).reshape(-1)
        score=metrics(val.label,p)
        row=dict(epoch=epoch+1,global_epoch=global_epoch+1,stage=stage,loss=loss,
            learning_rate=learning_rate,train_seconds=train_seconds,validation_seconds=time.perf_counter()-start,
            gpu=str(env['gpu']),**score,**probability_diagnostics(val.label,p))
        improved=score['macro_f1']>float(best_f1.numpy()) or (
            score['macro_f1']==float(best_f1.numpy()) and score['recall']>float(best_recall.numpy()))
        if improved:
            best_f1.assign(score['macro_f1']);best_recall.assign(score['recall']);best_epoch.assign(epoch+1);wait.assign(0)
            best_path.assign(best_manager.save(checkpoint_number=epoch+1))
        else:wait.assign_add(1)
        epoch_var.assign(epoch+1);history.append(row);history_var.assign(json.dumps(json_safe(history)))
        manager.save(checkpoint_number=epoch+1)
        pd.DataFrame(history).to_csv(out/'history.csv',index=False)
        print(f"{key} {stage} {epoch+1}/{budget}: macro-F1={score['macro_f1']:.4f}, AUC={score['roc_auc']:.4f}, p_std={row['probability_std']:.6f}",flush=True)
    selected=best_path.numpy().decode()
    if not selected:raise RuntimeError('No completed epoch in this stage')
    selected_weights.restore(selected).assert_existing_objects_matched()
    result=export_predictions(model,val,val_ds,out,'validation',key,meta['dataset'],protocol['training_seed'])
    result.update(best_epoch=int(best_epoch.numpy()),best_stage=stage,parameters=model.count_params(),
        trainable_parameters=int(sum(np.prod(v.shape) for v in model.trainable_variables)),
        total_train_minutes=sum(r['train_seconds'] for r in history)/60,epochs_completed=int(epoch_var.numpy()))
    save_json(out/'validation_metrics.json',result)
    pd.DataFrame(history).to_csv(out/'history.csv',index=False)
    state=dict(best_checkpoint=selected,last_checkpoint=manager.latest_checkpoint,stage=stage,
        initial_checkpoint=initial_checkpoint,validation_only=True)
    save_json(complete,state)
    del model;keras.backend.clear_session();gc.collect()
    return result,state

def run_model(key,frames,image_dir,meta,run_root,protocol):
    out=run_dir(run_root,meta['dataset'],key,protocol);out.mkdir(parents=True,exist_ok=True)
    config=experiment_config(key,meta['dataset'],protocol)
    path=out/'config.json'
    if path.exists() and json.loads(path.read_text())!=config:raise RuntimeError('Run config mismatch')
    save_json(path,config)
    if (out/'COMPLETE.json').exists():
        print(key,'already complete under V2')
        return json.loads((out/'validation_metrics.json').read_text()),out
    save_json(out/'environment.json',environment_record())
    audit=preprocessing_audit(frames,image_dir,meta,key,protocol)
    save_json(out/'preprocessing_audit.json',audit)
    head,hs=run_stage(key,frames,image_dir,meta,out/'head',protocol,'head')
    ft,fs=run_stage(key,frames,image_dir,meta,out/'finetune',protocol,'finetune',hs['best_checkpoint'])
    # Global validation selection; retain head-only result if fine-tuning does not improve it.
    choose_ft=(ft['macro_f1'],ft['recall'])>(head['macro_f1'],head['recall'])
    selected,state,stage=(ft,fs,'finetune') if choose_ft else (head,hs,'head')
    for filename in ['validation_predictions.csv','validation_confusion_matrix.png']:
        shutil.copyfile(out/stage/filename,out/filename)
    result=dict(selected)
    result['best_phase_epoch']=selected['best_epoch']
    result['best_epoch']=selected['best_epoch']+(protocol['head_epochs'] if choose_ft else 0)
    result['epochs_completed']=head['epochs_completed']+ft['epochs_completed']
    result['total_train_minutes']=head['total_train_minutes']+ft['total_train_minutes']
    save_json(out/'validation_metrics.json',result)
    histories=[pd.read_csv(out/s/'history.csv') for s in ['head','finetune']]
    pd.concat(histories,ignore_index=True).to_csv(out/'history.csv',index=False)
    save_json(out/'stage_comparison.json',dict(head=head,finetune=ft,selected_stage=stage))
    save_json(out/'COMPLETE.json',dict(best_checkpoint=state['best_checkpoint'],best_stage=stage,
        config_sha256=file_sha(path),validation_only=True))
    return result,out

def evaluate_frozen_test(key,frames,image_dir,meta,out,protocol):
    complete=json.loads((out/'COMPLETE.json').read_text())
    record=dict(config_sha256=file_sha(out/'config.json'),test_csv_sha256=meta['file_sha256']['test.csv'],
        best_checkpoint=complete['best_checkpoint'],threshold=.5,selection='validation macro-F1, tie positive recall')
    lock=out/'TEST_LOCK.json'
    if lock.exists() and json.loads(lock.read_text())!=record:raise RuntimeError('Test protocol changed')
    save_json(lock,record)
    if (out/'test_metrics.json').exists():return json.loads((out/'test_metrics.json').read_text())
    keras.backend.clear_session()
    model=build_model(key,meta['image_size'],protocol['training_seed'],weights=None,
        learning_rate=protocol['finetune_learning_rate'],stage='finetune')
    weight_snapshot(model).restore(complete['best_checkpoint']).assert_existing_objects_matched()
    ds=make_dataset(frames['test'],image_dir,key,protocol['batch_size'],protocol['training_seed'],size=meta['image_size'])
    result=export_predictions(model,frames['test'],ds,out,'test',key,meta['dataset'],protocol['training_seed'])
    del model;keras.backend.clear_session();gc.collect()
    return result

"""
exec(compile(TRAINER_SOURCE, "<B03_V2_TRAINER>", "exec"))
TRAINER_SHA256 = hashlib.sha256(TRAINER_SOURCE.encode("utf-8")).hexdigest()
print("Trainer SHA-256:", TRAINER_SHA256)

## 3. Konfigurasi
Default menguji dua model yang menunjukkan gejala collapse pada screening V1. Ini tidak otomatis membuktikan kesalahan arsitektur atau preprocessing. Setelah protokol diputuskan, ubah `MODELS_TO_RUN=list(REGISTRY)` untuk perbandingan lengkap. Model V2 yang sudah selesai akan dilewati.

Jangan ubah konfigurasi berdasarkan test. Parameter training berlaku sama untuk semua model. Angka epoch/learning rate berikut adalah titik awal eksperimen, bukan hasil pencarian parameter optimal.

In [ ]:
RESEARCH=Path('/content/drive/MyDrive/Research')
DATASET='cbis_ddsm'  # 'cbis_ddsm', 'cmmd', 'inbreast', 'mias'
MODELS_TO_RUN=['mobilenet_v2','vgg16']
# MODELS_TO_RUN=list(REGISTRY)  # All eight models under this SAME V2 protocol.
TRAINING_SEED=42
HEAD_EPOCHS=5
FINETUNE_EPOCHS=25
HEAD_LEARNING_RATE=1e-3
FINETUNE_LEARNING_RATE=1e-5
PATIENCE=5  # Fine-tuning only; head stage runs its fixed budget.
BATCH_SIZE=8
EVALUATE_TEST=False
PREP_ROOT=RESEARCH/'outputs/benchmark_4datasets/prepared_v1'
RUN_ROOT=RESEARCH/'outputs/benchmark_4datasets/benchmark_runs_v2_two_stage'
LOCAL_IMAGES=Path('/content/benchmark_b03_images')
assert DATASET in ['cbis_ddsm','cmmd','inbreast','mias']
assert MODELS_TO_RUN and all(k in REGISTRY for k in MODELS_TO_RUN)
assert HEAD_EPOCHS>0 and FINETUNE_EPOCHS>0 and PATIENCE>0 and BATCH_SIZE>0

## 4. Memuat cohort dan split B02C yang sama
Tidak perlu menjalankan ulang B01/B02C jika cache READY masih valid. Registry menentukan folder corrected yang digunakan. Hash CSV/ZIP dan piksel cache diverifikasi sebelum training. `negative` dan `positive` berarti label 0 dan 1, bukan hasil prediksi model.

CBIS-DDSM/CMMD: benign versus malignant sesuai cohort B02C. MIAS: subset abnormal benign versus malignant; normal tidak masuk tugas ini. INbreast: BI-RADS 1/2 versus 5/6 sebagai label proxy; bukan konfirmasi histopatologi. Kelompok INbreast berasal dari token filename pada pipeline ini dan belum diverifikasi secara independen sebagai identitas pasien. Jangan menyamakan arti label atau kepastian grouping seluruh dataset.

In [ ]:
ready_registry=json.loads((PREP_ROOT/'registry.json').read_text())
entry=ready_registry.get(DATASET,{})
if entry.get('status')!='READY':
    raise RuntimeError(f'{DATASET} is not READY: {entry}')
FRAMES,IMAGE_DIR,DATA_META=load_ready_dataset(Path(entry['path'])/'READY.json',LOCAL_IMAGES)
PROTOCOL=dict(training_seed=TRAINING_SEED,split_seed=2026,batch_size=BATCH_SIZE,
    head_epochs=HEAD_EPOCHS,finetune_epochs=FINETUNE_EPOCHS,patience=PATIENCE,
    head_learning_rate=HEAD_LEARNING_RATE,finetune_learning_rate=FINETUNE_LEARNING_RATE,
    image_size=DATA_META['image_size'],pipeline_version=DATA_META['pipeline_version'],
    cohort_sha256=DATA_META['cohort_sha256'],images_sha256=DATA_META['file_sha256']['images.zip'],
    data_file_sha256=DATA_META['file_sha256'],trainer_sha256=TRAINER_SHA256,
    trainer_patch='two_stage_frozen_bn_nonfused_v2',framework='tf_keras',
    tf_version=tf.__version__,tf_keras_version=keras.__version__,precision='float32',weights='imagenet',
    architecture_registry={k:list(v) for k,v in REGISTRY.items()},
    head='GAP_dropout0.2_sigmoid',batch_norm='frozen',batch_norm_implementation='non_fused',
    augmentation='horizontal_flip',class_weight='balanced_from_train_only',threshold=0.5,
    checkpoint_selection='validation macro_f1, tie positive recall; ties across stages retain head',
    fine_tune_start='best head checkpoint, fresh Adam optimizer',head_stopping='fixed epoch budget')
PROTOCOL_DIR=protocol_dir(RUN_ROOT,DATASET,PROTOCOL)
PROTOCOL_DIR.mkdir(parents=True,exist_ok=True)
save_json(PROTOCOL_DIR/'protocol.json',PROTOCOL)
(PROTOCOL_DIR/'trainer_source.py').write_text(TRAINER_SOURCE,encoding='utf-8')
save_json(PROTOCOL_DIR/'data_provenance.json',dict(ready_path=str(Path(entry['path'])/'READY.json'),metadata=DATA_META))
save_json(PROTOCOL_DIR/'runtime.json',environment_record())
display(pd.DataFrame([{'split':s,'images':len(d),'patient_groups':d.patient_id.nunique(),
    'negative':int(d.label.eq(0).sum()),'positive':int(d.label.eq(1).sum())} for s,d in FRAMES.items()]))
print('Output:',PROTOCOL_DIR)
print('GPU details:',environment_record()['gpu'])

## 5. Training dan validasi
Setiap fase menyimpan checkpoint terakhir, optimizer, epoch, patience, histori, dan checkpoint terbaik. Jika terputus, jalankan notebook dari atas dengan konfigurasi sama. Epoch yang belum tersimpan akan diulang; dropout RNG dan runtime GPU dapat membuat hasil resume tidak identik bit-per-bit dengan run tanpa interupsi.

Audit preprocessing memeriksa satu batch train terhadap rumus normalisasi masing-masing keluarga model. Histori juga merekam rentang/SD probabilitas dan proporsi prediksi positif. Probabilitas nyaris konstan atau semua prediksi satu kelas adalah sinyal diagnosis; bukan alasan mengubah threshold menggunakan test.

Waktu training adalah jumlah waktu `fit` kedua fase, tidak mencakup download ImageNet, persiapan data, validasi, atau penulisan checkpoint. Semua bobot backbone dimulai dari ImageNet saat run baru; head baru diinisialisasi. Checkpoint terakhir digunakan untuk resume.

In [ ]:
CURRENT_RESULTS=[]
for key in MODELS_TO_RUN:
    print('TRAINING V2',DATASET,key,'seed',TRAINING_SEED,flush=True)
    result,out=run_model(key,FRAMES,IMAGE_DIR,DATA_META,RUN_ROOT,PROTOCOL)
    CURRENT_RESULTS.append(result)
display(pd.DataFrame(CURRENT_RESULTS)[['model','role','precision','recall','f1','macro_f1',
    'accuracy','roc_auc','best_stage','best_epoch','parameters','total_train_minutes']].round(4))

## 6. Ringkasan seluruh model yang selesai dalam protokol ini
Precision/recall/F1 adalah metrik kelas positif. Macro-F1 merata-ratakan F1 dua kelas dan menjadi aturan pemilihan checkpoint/kandidat yang ditetapkan sejak V1. Accuracy dan ROC-AUC tetap ditampilkan sebagai metrik tambahan. Jika Teacher dan Student dipilih, pilih terpisah berdasarkan hasil validasi; jika skor dan recall sama, laporkan seri.

Hasil berikut masih VALIDATION, bukan TEST. Screening satu seed dan pencobaan beberapa protokol pada validation membuat skor validation berpotensi optimistis.

In [ ]:
COMPLETED={}
for key in REGISTRY:
    folder=run_dir(RUN_ROOT,DATASET,key,PROTOCOL)
    if (folder/'COMPLETE.json').exists():
        if json.loads((folder/'config.json').read_text())!=experiment_config(key,DATASET,PROTOCOL):
            raise RuntimeError('Saved configuration mismatch')
        COMPLETED[key]=folder
all_results=pd.DataFrame([json.loads((folder/'validation_metrics.json').read_text()) for folder in COMPLETED.values()])
if all_results.empty: raise RuntimeError('No completed V2 models yet')
all_results.to_csv(PROTOCOL_DIR/'validation_summary.csv',index=False)
display(all_results.sort_values(['role','macro_f1','recall'],ascending=[True,False,False]).round(4))
print('Completed:',len(COMPLETED),'/ 8 models in V2')
for key,folder in COMPLETED.items():
    comparison=json.loads((folder/'stage_comparison.json').read_text())
    display(pd.DataFrame([dict(stage=s,**{m:comparison[s][m] for m in ['precision','recall','f1','macro_f1','roc_auc','best_epoch']})
                          for s in ['head','finetune']]).assign(model=key).round(4))
    print(key,'selected stage:',comparison['selected_stage'])

## 7. Test setelah seluruh delapan model selesai dan protokol dibekukan
Default tetap `EVALUATE_TEST=False`. Saat tidak ada lagi tuning protokol, ubah menjadi True lalu jalankan kembali cell konfigurasi, pemuatan, ringkasan, dan cell ini. Semua delapan model V2 harus selesai. Seleksi kandidat dari validation disimpan sebelum membuka hasil test; jangan mengganti seleksi berdasarkan test.

Test tidak digunakan untuk early stopping atau memilih threshold. Evaluasi ini within-dataset; bukan validasi generalisasi lintas dataset. Tidak ada KD/MoE pada tahap benchmark ini.

In [ ]:
if EVALUATE_TEST:
    if set(COMPLETED)!=set(REGISTRY):
        raise RuntimeError('Complete the eight modules in this protocol before taking the test.')
    selection={}
    for role,g in all_results.groupby('role'):
        best=g.sort_values(['macro_f1','recall'],ascending=False).iloc[0]
        selection[role]=g.loc[g.macro_f1.eq(best.macro_f1)&g.recall.eq(best.recall),'model'].tolist()
    freeze={'protocol':PROTOCOL,'selection_from_validation':selection,'models':list(REGISTRY)}
    freeze_path=PROTOCOL_DIR/'FROZEN_BEFORE_TEST.json'
    if freeze_path.exists() and json.loads(freeze_path.read_text())!=freeze:
        raise RuntimeError('Previously frozen benchmark differs')
    save_json(freeze_path,freeze)
    tests=[]
    for key,folder in COMPLETED.items():
        tests.append(evaluate_frozen_test(key,FRAMES,IMAGE_DIR,DATA_META,folder,PROTOCOL))
    test_summary=pd.DataFrame(tests)
    test_summary.to_csv(PROTOCOL_DIR/'test_summary.csv',index=False)
    display(test_summary[['model','role','precision','recall','f1','macro_f1','accuracy','roc_auc']].round(4))
else:
    print('The test has not yet been evaluated. The current result is VALIDATION.')

## 8. Unduh laporan V2
ZIP berisi konfigurasi, kode trainer yang dijalankan, provenance data, environment, audit preprocessing, histori kedua tahap, prediksi, metrik, dan confusion matrix biru. Bobot besar tetap di Drive. Kirim ZIP ini setelah dua model awal selesai; nama laporan menyertakan V2 dan signature protokol agar tidak tertukar dengan V1.

In [ ]:
report=PROTOCOL_DIR/f'B03_V2_{DATASET}_seed_{TRAINING_SEED}_{PROTOCOL_DIR.name}_report.zip'
with zipfile.ZipFile(report,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in COMPLETED.values():
        for f in folder.rglob('*'):
            if f.is_file() and f.suffix in ['.json','.csv','.png']:
                z.write(f,str(f.relative_to(PROTOCOL_DIR)))
    for f in PROTOCOL_DIR.iterdir():
        if f.is_file() and f.suffix in ['.json','.csv','.py']:
            z.write(f,f.name)
print(report)
from google.colab import files
files.download(str(report))

## Acuan dan batas verifikasi
- Panduan resmi Keras tentang head training, kemudian fine-tuning dengan learning rate rendah: https://keras.io/guides/transfer_learning/
- Preprocessing MobileNetV2 dan MobileNetV3: https://keras.io/2/api/applications/mobilenet/
- Implementasi TF-Keras: https://github.com/keras-team/tf-keras

Notebook ini diperiksa sintaksnya; delapan rumus preprocessing diuji pada citra sintetis. Alur dua tahap, resume setelah interupsi, pemilihan tahap terbaik, skip run selesai, dan pemulihan bobot untuk test diuji dengan jaringan kecil pada CPU. MobileNetV2 aktual juga diuji pemindahan bobot antartahap dan satu langkah gradien non-fused pada CPU, memakai bobot acak. Pelatihan delapan arsitektur dengan bobot ImageNet pada citra Anda dan GPU Colab belum dijalankan di lingkungan pemeriksaan ini. Hasil eksperimen nyata harus diverifikasi dari laporan V2; perbaikan performa tidak dijamin.

Pertahankan catatan hasil V1 beserta perubahan protokol. Jangan membandingkan angka terbaik dari protokol berbeda seolah-olah berasal dari satu eksperimen yang sama.